<a href="https://colab.research.google.com/github/yashwanthks78/pytorch/blob/main/Simple_RNN_new.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

In [ ]:
corpus = [
    "I love machine learning",
    "word2vec is great algorithm",
    "Implementing word2vec is fun"
]

In [ ]:

sentences = [s.lower().split() for s in corpus]

In [ ]:
sentences

[['i', 'love', 'machine', 'learning'],
 ['word2vec', 'is', 'great', 'algorithm'],
 ['implementing', 'word2vec', 'is', 'fun']]

In [ ]:
vocab = sorted(set(word for sent in sentences for word in sent))

In [ ]:
vocab

['algorithm',
 'fun',
 'great',
 'i',
 'implementing',
 'is',
 'learning',
 'love',
 'machine',
 'word2vec']

In [ ]:
word2idx = {w:i for i,w in enumerate(vocab)}
# idx2word = {i:w for w,i in word2idx.items()}
idx2word = {i:w for i,w in enumerate(vocab)}

In [ ]:
idx2word

{0: 'algorithm',
 1: 'fun',
 2: 'great',
 3: 'i',
 4: 'implementing',
 5: 'is',
 6: 'learning',
 7: 'love',
 8: 'machine',
 9: 'word2vec'}

In [ ]:
vocab_size = len(vocab)

In [ ]:
word2idx

{'algorithm': 0,
 'fun': 1,
 'great': 2,
 'i': 3,
 'implementing': 4,
 'is': 5,
 'learning': 6,
 'love': 7,
 'machine': 8,
 'word2vec': 9}

In [ ]:
X = []
Y = []
for sent in sentences:
    for i in range(len(sent)-1):
        X.append(word2idx[sent[i]])
        Y.append(word2idx[sent[i+1]])

In [ ]:
X = torch.tensor(X)
Y = torch.tensor(Y)

In [ ]:
print(X)
print(Y)

tensor([3, 7, 8, 9, 5, 2, 4, 9, 5])
tensor([7, 8, 6, 5, 2, 0, 9, 5, 1])


In [ ]:
vocab_size

10

In [ ]:

class NextWordRNN(nn.Module):
    def __init__(self, vocab_size, embed_dim=100, hidden_dim=16):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.rnn = nn.RNN(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x):
        x = self.embedding(x) # [9, 1, 100]
        out, hx = self.rnn(x) # [9, 1, 16], [1, 9, 16]
        out = out[:, -1, :] # [9, 16]
        out = self.fc(out) # [9, 10]
        return out


In [ ]:
model = NextWordRNN(vocab_size)

In [ ]:
# inputs = X.unsqueeze(1)
# outputs = model(inputs)

torch.Size([9, 1, 100])
torch.Size([9, 1, 16]) torch.Size([1, 9, 16])
torch.Size([9, 16])
torch.Size([9, 10])


In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

In [ ]:

for epoch in range(300):

    optimizer.zero_grad()

    inputs = X.unsqueeze(1)
    outputs = model(inputs)

    loss = loss_fn(outputs, Y)

    loss.backward()
    optimizer.step()

    if epoch % 50 == 0:
        print("Epoch:", epoch, "Loss:", loss.item())

Epoch: 0 Loss: 2.388366222381592
Epoch: 50 Loss: 0.196868896484375
Epoch: 100 Loss: 0.16553625464439392
Epoch: 150 Loss: 0.16021747887134552
Epoch: 200 Loss: 0.15799380838871002
Epoch: 250 Loss: 0.15682119131088257


In [ ]:
def predict_next(word):
    model.eval()
    idx = torch.tensor([[word2idx[word.lower()]]])

    with torch.no_grad():
        out = model(idx)
        pred = torch.argmax(out).item()

    return idx2word[pred]

In [ ]:
print("machine=>", predict_next("machine"))
print("is=>", predict_next("is"))
print("word2vec=>", predict_next("word2vec"))

machine=> learning
is=> great
word2vec=> is
